# 02 - Model Training and Comparison

Este notebook entrena y compara varios modelos sencillos de clasificación para predecir churn en el dataset sintético SaaS.

Objetivos:
- preparar los datos para modelado,
- entrenar varios modelos básicos y comparables,
- evaluar su rendimiento,
- comparar resultados,
- identificar un modelo final razonable para el prototipo.


## 1. Importación de librerías

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
    classification_report,
    RocCurveDisplay
)

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)
RANDOM_STATE = 42


## 2. Carga del dataset

Ajusta la ruta si es necesario.


In [ ]:
file_path = 'synthetic_saas_churn_dataset.csv'
df = pd.read_csv(file_path)

print('Shape:', df.shape)
df.head()


## 3. Selección de variables

Se excluye `account_id` porque es solo un identificador.
La variable objetivo es `churn_label`.


In [ ]:
target = 'churn_label'
drop_cols = ['account_id']

X = df.drop(columns=drop_cols + [target])
y = df[target]

categorical_features = X.select_dtypes(include=['object']).columns.tolist()
numeric_features = X.select_dtypes(exclude=['object']).columns.tolist()

print('Variables categóricas:', categorical_features)
print('Variables numéricas:', numeric_features)


## 4. División train/test

Se utiliza una división simple para reservar una parte de los datos para evaluación final.


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.25,
    random_state=RANDOM_STATE,
    stratify=y
)

print('Train shape:', X_train.shape)
print('Test shape:', X_test.shape)
print('Churn train:', round(y_train.mean(), 4))
print('Churn test:', round(y_test.mean(), 4))


## 5. Preprocesamiento

- Variables numéricas: imputación simple y escalado.
- Variables categóricas: imputación y codificación one-hot.


In [ ]:
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ]
)

preprocessor


## 6. Definición de modelos

Se prueban tres modelos:
- Regresión logística: referencia simple e interpretable.
- Random Forest: más flexible.
- Gradient Boosting: versión más robusta para comparar.


In [ ]:
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
    'Random Forest': RandomForestClassifier(
        n_estimators=250,
        max_depth=8,
        min_samples_leaf=3,
        random_state=RANDOM_STATE
    ),
    'Gradient Boosting': GradientBoostingClassifier(
        n_estimators=150,
        learning_rate=0.08,
        random_state=RANDOM_STATE
    )
}


## 7. Entrenamiento y evaluación

Se calculan las métricas principales:
- Accuracy
- Precision
- Recall
- F1-score
- ROC-AUC


In [ ]:
results = []
fitted_pipelines = {}

for model_name, model in models.items():
    pipeline = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('model', model)
    ])

    pipeline.fit(X_train, y_train)
    fitted_pipelines[model_name] = pipeline

    y_pred = pipeline.predict(X_test)
    y_prob = pipeline.predict_proba(X_test)[:, 1]

    row = {
        'model': model_name,
        'accuracy': accuracy_score(y_test, y_pred),
        'precision': precision_score(y_test, y_pred, zero_division=0),
        'recall': recall_score(y_test, y_pred, zero_division=0),
        'f1_score': f1_score(y_test, y_pred, zero_division=0),
        'roc_auc': roc_auc_score(y_test, y_prob)
    }
    results.append(row)

results_df = pd.DataFrame(results).sort_values(by='f1_score', ascending=False).reset_index(drop=True)
results_df


## 8. Comparación visual de resultados

In [ ]:
plot_df = results_df.set_index('model')[['accuracy', 'precision', 'recall', 'f1_score', 'roc_auc']]

ax = plot_df.plot(kind='bar', figsize=(10,5))
ax.set_title('Comparación de modelos')
ax.set_ylabel('Score')
ax.set_xlabel('Modelo')
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()


## 9. Matrices de confusión

In [ ]:
for model_name, pipeline in fitted_pipelines.items():
    y_pred = pipeline.predict(X_test)
    cm = confusion_matrix(y_test, y_pred)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm)
    disp.plot()
    plt.title(f'Matriz de confusión - {model_name}')
    plt.tight_layout()
    plt.show()


## 10. Curvas ROC

In [ ]:
plt.figure(figsize=(8,6))

for model_name, pipeline in fitted_pipelines.items():
    RocCurveDisplay.from_estimator(pipeline, X_test, y_test, name=model_name, ax=plt.gca())

plt.title('Curvas ROC por modelo')
plt.tight_layout()
plt.show()


## 11. Classification report del mejor modelo

In [ ]:
best_model_name = results_df.iloc[0]['model']
best_pipeline = fitted_pipelines[best_model_name]

y_pred_best = best_pipeline.predict(X_test)
print('Mejor modelo:', best_model_name)
print()
print(classification_report(y_test, y_pred_best, digits=3))


## 12. Importancia de variables

Para modelos basados en árboles, se extraen importancias aproximadas.


In [ ]:
for model_name in ['Random Forest', 'Gradient Boosting']:
    pipeline = fitted_pipelines[model_name]
    model = pipeline.named_steps['model']
    prep = pipeline.named_steps['preprocessor']

    feature_names = prep.get_feature_names_out()
    importances = model.feature_importances_

    importance_df = pd.DataFrame({
        'feature': feature_names,
        'importance': importances
    }).sort_values(by='importance', ascending=False).head(15)

    print(f'\nTop variables - {model_name}')
    print(importance_df)

    plt.figure(figsize=(8,5))
    plt.barh(importance_df['feature'][::-1], importance_df['importance'][::-1])
    plt.title(f'Top 15 variables más relevantes - {model_name}')
    plt.xlabel('Importancia')
    plt.tight_layout()
    plt.show()


## 13. Guardado del mejor modelo

Se guarda el pipeline completo para usarlo después en el prototipo.


In [ ]:
joblib.dump(best_pipeline, 'best_churn_model.pkl')
print('Modelo guardado como best_churn_model.pkl')


## 14. Tabla resumen para el TFM

In [ ]:
summary_for_report = results_df.copy()
metric_cols = ['accuracy', 'precision', 'recall', 'f1_score', 'roc_auc']
summary_for_report[metric_cols] = summary_for_report[metric_cols].round(3)
summary_for_report


## 15. Conclusiones iniciales

Puedes adaptar estas ideas a la memoria:

- La primera aproximación sirve como línea base para validar el problema.
- Los modelos más flexibles mejoran la detección de cuentas con señales de riesgo más complejas.
- El mejor modelo se selecciona no solo por su rendimiento, sino también por su utilidad práctica para el prototipo.
- Las variables relacionadas con engagement, renovación, satisfacción y sentimiento tienden a aparecer entre las más relevantes.
